In [0]:
%pip install pymupdf

In [0]:
dbutils.library.restartPython()

### For PDF (discharge_summaries)

In [0]:
# Read PDF files from the Volume
from pyspark.sql import functions as F

PDF_PATH = "/Volumes/claims/raw/landing/genai/discharge_summaries/"

pdf_df = (
    spark.read
    .format("binaryFile")
    .option("pathGlobFilter", "*.pdf")
    .load(PDF_PATH)
    .select(
        F.col("path"),
        F.col("content"),
        F.element_at(
            F.split(F.col("path"), "/"), -1
        ).alias("source_file")
    )
)

print("Total PDF files:", pdf_df.count())

display(pdf_df.select("path", "source_file"))

In [0]:
# Parse the PDF using AI Functions
parsed_df = (
    pdf_df
    .withColumn(
        "parsed_document",
        F.expr("""
            ai_parse_document(
                content,
                map('version', '2.0')
            )
        """)
    )
)

display(
    parsed_df.select(
        "source_file",
        "parsed_document"
    )
)

In [0]:
# Extract the complete document text
parsed_text_df = parsed_df.withColumn(
    "document_text",
    F.expr("""
        concat_ws(
            '\\n',
            transform(
                from_json(
                    to_json(parsed_document:document:elements),
                    'array<struct<content:string>>'
                ),
                x -> x.content
            )
        )
    """)
)

display(
    parsed_text_df.select(
        "source_file",
        "document_text"
    )
)

In [0]:
# Define the AI extraction schema
extraction_schema = {
    "claim_id": {
        "type": "string",
        "description": "Claim reference from the discharge summary, such as CLM-10072."
    },
    "diagnosis_code": {
        "type": "string",
        "description": "ICD-10 code of the FINAL diagnosis only. Ignore older, historical, unrelated, or provisional diagnoses."
    },
    "procedure_code": {
        "type": "string",
        "description": "Procedure code corresponding to the procedure performed during this hospital admission."
    },
    "admission_date": {
        "type": "string",
        "description": "Date of admission in YYYY-MM-DD format."
    },
    "discharge_date": {
        "type": "string",
        "description": "Date of discharge in YYYY-MM-DD format."
    },
    "room_type": {
        "type": "enum",
        "description": "Normalize the room category to one of: general_ward, twin_sharing, single_private, deluxe.",
        "values": [
            "general_ward",
            "twin_sharing",
            "single_private",
            "deluxe"
        ]
    }
}

In [0]:
# Run ai_extract()
from pyspark.sql.functions import ai_extract

# ai_extract expects 'labels' (not 'values') for enum-type fields
for field_schema in extraction_schema.values():
    if "values" in field_schema:
        field_schema["labels"] = field_schema.pop("values")

extracted_df = (
    parsed_text_df
    .withColumn(
        "ai_result",
        ai_extract(
            F.col("parsed_document"),
            extraction_schema,
            {
                "version": "2.1",
                "instructions": """
                    You are extracting structured information
                    from Saffron Shield hospital discharge summaries.

                    Extract the claim reference and five required facts.

                    IMPORTANT RULES:

                    1. Extract the ICD-10 code of the FINAL diagnosis.
                    2. Ignore older, unrelated, historical, or provisional diagnoses.
                    3. Extract the procedure code of the procedure actually performed.
                    4. Extract admission and discharge dates accurately.
                    5. Normalize room types to the supplied enum values.
                    6. Do not invent missing information.
                    7. Return null if a field cannot be identified.
                """
            }
        )
    )
)

display(
    extracted_df.select(
        "source_file",
        "ai_result"
    )
)

In [0]:
# Convert AI results into the required columns
result_df = (
    extracted_df
    .withColumn("result_json", F.to_json("ai_result"))
    .select(
        F.get_json_object(
            "result_json", "$.response.claim_id.value"
        ).alias("claim_id"),

        "source_file",

        F.lit("pdf").alias("source"),

        "document_text",

        F.get_json_object(
            "result_json", "$.response.diagnosis_code.value"
        ).alias("diagnosis_code"),

        F.get_json_object(
            "result_json", "$.response.procedure_code.value"
        ).alias("procedure_code"),

        F.to_date(
            F.get_json_object(
                "result_json", "$.response.admission_date.value"
            )
        ).alias("admission_date"),

        F.to_date(
            F.get_json_object(
                "result_json", "$.response.discharge_date.value"
            )
        ).alias("discharge_date"),

        F.get_json_object(
            "result_json", "$.response.room_type.value"
        ).alias("room_type")
    )
)

display(result_df)

In [0]:
(
    result_df
    .select(
        "claim_id",
        "source_file",
        "source",
        "document_text",
        "diagnosis_code",
        "procedure_code",
        "admission_date",
        "discharge_date",
        "room_type"
    )
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("claims.genai.discharge_extracted")
)

In [0]:
%sql
select * from claims.genai.discharge_extracted

In [0]:
%sql

WITH comparison AS (
    SELECT
        e.claim_id,

        CASE
            WHEN UPPER(TRIM(e.diagnosis_code))
               = UPPER(TRIM(c.diagnosis_code))
            THEN 1 ELSE 0
        END AS diagnosis_match,

        CASE
            WHEN UPPER(TRIM(e.procedure_code))
               = UPPER(TRIM(c.procedure_code))
            THEN 1 ELSE 0
        END AS procedure_match,

        CASE
            WHEN e.admission_date = c.admission_date
            THEN 1 ELSE 0
        END AS admission_match,

        CASE
            WHEN e.discharge_date = c.discharge_date
            THEN 1 ELSE 0
        END AS discharge_match

    FROM claims.genai.discharge_extracted e

    INNER JOIN claims.silver.claims_clean c
        ON e.claim_id = c.claim_id
),

scored AS (
    SELECT
        *,
        diagnosis_match +
        procedure_match +
        admission_match +
        discharge_match AS matched_fields
    FROM comparison
)

SELECT
    COUNT(*) AS compared_claims,
    SUM(matched_fields) AS matching_fields,
    COUNT(*) * 4 AS total_fields,

    ROUND(
        100.0 * SUM(matched_fields)
        / NULLIF(COUNT(*) * 4, 0),
        2
    ) AS match_percentage

FROM scored;

### For Txt (discharge_summaries_txt)

In [0]:
# # Configure the input path
# from pyspark.sql import functions as F
# from pyspark.sql.types import *
# from datetime import datetime
# import re
# import os

# # Use TXT files so every output row has source = 'txt'
# SOURCE_FORMAT = "txt"

# INPUT_PATH = (
#     "/Volumes/claims/raw/landing/genai/"
#     "discharge_summaries_txt/"
# )

# TARGET_TABLE = "claims.genai.discharge_extracted"

In [0]:
# # Create extraction functions
# def extract_field(pattern, text):
#     match = re.search(
#         pattern,
#         text,
#         re.IGNORECASE | re.MULTILINE
#     )

#     return match.group(1).strip() if match else None


# def parse_date(date_value):
#     if not date_value:
#         return None

#     formats = [
#         "%b %d, %Y",       # Jan 20, 2026
#         "%B %d, %Y",       # January 20, 2026
#         "%d/%m/%Y",
#         "%d-%m-%Y",
#         "%Y-%m-%d",
#         "%d %B %Y"
#     ]

#     for fmt in formats:
#         try:
#             return datetime.strptime(date_value.strip(), fmt).date()
#         except ValueError:
#             continue

#     return None


# def extract_room_type(room_value):
#     if not room_value:
#         return None

#     value = room_value.lower()

#     if "general ward" in value:
#         return "general_ward"

#     if "twin sharing" in value:
#         return "twin_sharing"

#     if "single private" in value:
#         return "single_private"

#     if "deluxe" in value:
#         return "deluxe"

#     return None


# def extract_discharge_summary(document_text, source_file):

#     # Claim reference
#     claim_id = extract_field(
#         r"Claim\s+reference\s*:\s*([A-Z0-9-]+)",
#         document_text
#     )

#     # Final diagnosis ICD-10 code
#     diagnosis_code = extract_field(
#         r"Final\s+diagnosis\s*:.*?\(ICD-10\s+([A-Z0-9.]+)\)",
#         document_text
#     )

#     # Procedure code
#     procedure_code = extract_field(
#         r"Procedure\s+performed\s*:.*?\(procedure\s+code\s+([A-Z0-9-]+)\)",
#         document_text
#     )

#     # Admission date
#     admission_raw = extract_field(
#         r"Date\s+of\s+admission\s*:\s*([^\r\n]+)",
#         document_text
#     )

#     # Discharge date
#     discharge_raw = extract_field(
#         r"Date\s+of\s+discharge\s*:\s*([^\r\n]+)",
#         document_text
#     )

#     # Room category
#     room_raw = extract_field(
#         r"Room\s+category\s*:\s*([^\r\n(]+)",
#         document_text
#     )

#     return {
#         "claim_id": claim_id,
#         "source_file": source_file,
#         "source": SOURCE_FORMAT,
#         "document_text": document_text,
#         "diagnosis_code": diagnosis_code,
#         "procedure_code": procedure_code,
#         "admission_date": parse_date(admission_raw),
#         "discharge_date": parse_date(discharge_raw),
#         "room_type": extract_room_type(room_raw)
#     }

In [0]:
# # Read all TXT files from your Unity Catalog Volume
# files_df = (
#     spark.read
#     .format("binaryFile")
#     .option("pathGlobFilter", "*.txt")
#     .load(INPUT_PATH)
#     .select("path", "content")
# )

# extracted_rows = []

# for row in files_df.toLocalIterator():

#     try:
#         document_text = row["content"].decode(
#             "utf-8-sig",
#             errors="replace"
#         )

#         source_file = row["path"].rsplit("/", 1)[-1]

#         result = extract_discharge_summary(
#             document_text,
#             source_file
#         )

#         extracted_rows.append(result)

#     except Exception as e:
#         print(
#             f"Error processing {row['path']}: {str(e)}"
#         )

# print("Total documents extracted:", len(extracted_rows))

In [0]:
# # Create the DataFrame
# output_schema = StructType([
#     StructField("claim_id", StringType(), True),
#     StructField("source_file", StringType(), True),
#     StructField("source", StringType(), False),
#     StructField("document_text", StringType(), True),
#     StructField("diagnosis_code", StringType(), True),
#     StructField("procedure_code", StringType(), True),
#     StructField("admission_date", DateType(), True),
#     StructField("discharge_date", DateType(), True),
#     StructField("room_type", StringType(), True)
# ])

# extracted_df = spark.createDataFrame(
#     extracted_rows,
#     schema=output_schema
# )

# display(extracted_df)

In [0]:
# # Write to claims.genai.discharge_extracted
# (
#     extracted_df
#     .write
#     .format("delta")
#     .mode("append")
#     .option("overwriteSchema", "true")
#     .saveAsTable(TARGET_TABLE)
# )

In [0]:
%sql
CREATE OR REPLACE TABLE claims.genai.grievance_emails AS
SELECT *
FROM read_files(
    '/Volumes/claims/raw/landing/genai/grievance_emails.csv',
    format => 'csv',
    header => true,
    inferSchema => true
);

In [0]:
%sql
select * from claims.genai.grievance_emails

In [0]:
%sql
CREATE OR REPLACE TABLE claims.genai.grievance_classified AS

WITH classified AS (
    SELECT
        email_id,

        ai_classify(
            CONCAT(
                'Subject: ', COALESCE(subject, ''),
                '\nEmail body: ', COALESCE(body, '')
            ),

            '{
                "claim_delay": "The customer complains about claim processing, approval, or settlement taking too long, or a claim remaining pending without an update.",
                "claim_rejection": "The customer disputes a claim rejection, denial, or refusal by the insurer, including rejection due to waiting periods or policy conditions.",
                "payout_deduction": "The customer complains about deductions, reduced settlement amounts, short payments, or receiving less money than expected.",
                "cashless_network": "The customer complains about cashless hospital admission, network hospital availability, or cashless authorization issues.",
                "policy_service": "The customer complains about policy servicing, renewal, policy documents, corrections, member details, or general policy administration."
            }',

            MAP(
                'version', '2.1',
                'instructions',
                'Classify each email into exactly one category based on its primary complaint. Consider both the subject and body. Ignore secondary or incidental issues. If the customer disputes a rejection, choose claim_rejection. If the complaint is about a pending claim or delayed processing, choose claim_delay. If the complaint is about a reduced payout or deduction, choose payout_deduction. If it concerns cashless hospital access, choose cashless_network. If it concerns general policy administration, choose policy_service. Return the single best matching category.',
                'enableConfidenceScores', 'true',
                'enableRationales', 'true'
            )
        ) AS classification

    FROM claims.genai.grievance_emails
)

SELECT
    email_id,
    get_json_object(
        to_json(classification),
        '$.response[0].value'
    ) AS category
FROM classified;

### R2 - Chunk, Embed and Index

In [0]:

import re
from pyspark.sql import functions as F
from delta.tables import DeltaTable

# ============================================================
# CONFIGURATION
# ============================================================

SOURCE_PATH = "/Volumes/claims/raw/landing/genai/policy_docs/"
TARGET_TABLE = "claims.genai.policy_chunks"

# ============================================================
# 1. READ ALL MARKDOWN POLICY DOCUMENTS
# ============================================================

files_df = (
    spark.read
    .format("binaryFile")
    .option("pathGlobFilter", "*.md")
    .load(SOURCE_PATH)
    .select("path", "content")
)

# Collect document contents to the driver.
# Appropriate for this small binder of eight documents.
documents = files_df.collect()

print(f"Total policy documents found: {len(documents)}")


# ============================================================
# 2. HELPER FUNCTION TO GENERATE STABLE CHUNK ID
# ============================================================

def generate_chunk_id(filename, heading):

    # Lowercase the heading
    anchor = heading.lower()

    # Replace every run of non-alphanumeric characters with "-"
    anchor = re.sub(r"[^a-z0-9]+", "-", anchor)

    # Remove leading and trailing "-"
    anchor = anchor.strip("-")

    # Filename keeps original case and contains no folder
    return f"{filename}#{anchor}"


# ============================================================
# 3. SPLIT DOCUMENT INTO SECTIONS
# ============================================================

def split_into_chunks(filename, document_text):

    # Normalize line endings
    document_text = document_text.replace("\r\n", "\n").replace("\r", "\n")

    # Find every Markdown ## heading
    heading_pattern = re.compile(r"(?m)^##[ \t]+(.+?)[ \t]*$")

    matches = list(heading_pattern.finditer(document_text))

    chunks = []

    # Document header = everything before first ##
    if not matches:
        print(f"WARNING: No ## sections found in {filename}")
        return chunks

    document_header = document_text[:matches[0].start()].strip()

    for index, match in enumerate(matches):

        heading = match.group(1).strip()

        # Section body extends until the next ## heading or EOF
        section_start = match.end()

        if index + 1 < len(matches):
            section_end = matches[index + 1].start()
        else:
            section_end = len(document_text)

        section_body = document_text[section_start:section_end].strip()

        # Construct exact chunk text:
        # document header
        # blank line
        # ## heading
        # blank line
        # section body

        heading_line = f"## {heading}"

        chunk_text = (
            f"{document_header}\n\n"
            f"{heading_line}\n\n"
            f"{section_body}"
        )

        # Filename without folder
        chunk_id = generate_chunk_id(filename, heading)

        chunks.append({
            "chunk_id": chunk_id,
            "doc_name": filename,
            "heading": heading,
            "chunk_index": index,
            "text": chunk_text
        })

    return chunks


# ============================================================
# 4. PROCESS ALL DOCUMENTS
# ============================================================

all_chunks = []

for document in documents:

    # Extract filename without folder
    filename = document["path"].rsplit("/", 1)[-1]

    # Decode PDF-independent Markdown text
    document_text = bytes(document["content"]).decode("utf-8")

    document_chunks = split_into_chunks(
        filename,
        document_text
    )

    all_chunks.extend(document_chunks)

    print(
        f"{filename}: {len(document_chunks)} sections extracted"
    )


print(f"\nTotal chunks generated: {len(all_chunks)}")


# ============================================================
# 5. CREATE SPARK DATAFRAME
# ============================================================

schema = """
    chunk_id STRING,
    doc_name STRING,
    heading STRING,
    chunk_index INT,
    text STRING
"""

chunks_df = spark.createDataFrame(
    all_chunks,
    schema=schema
)


# ============================================================
# 6. VALIDATE GENERATED CHUNKS BEFORE WRITING
# ============================================================

# Check duplicate chunk IDs
duplicate_ids = (
    chunks_df
    .groupBy("chunk_id")
    .count()
    .filter(F.col("count") > 1)
)

if duplicate_ids.limit(1).count() > 0:
    display(duplicate_ids)
    raise ValueError(
        "Duplicate chunk IDs found. Resolve duplicate headings "
        "within documents before loading the table."
    )


# Check required fields
invalid_chunks = chunks_df.filter(
    F.col("chunk_id").isNull()
    | F.col("doc_name").isNull()
    | F.col("heading").isNull()
    | F.col("text").isNull()
)

if invalid_chunks.limit(1).count() > 0:
    display(invalid_chunks)
    raise ValueError("Invalid chunks found. Aborting load.")

In [0]:
# ============================================================
# 7. CREATE TARGET TABLE IF IT DOES NOT EXIST
# ============================================================

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TARGET_TABLE} (
        chunk_id STRING NOT NULL,
        doc_name STRING,
        heading STRING,
        chunk_index INT,
        text STRING
    )
    USING DELTA
""")


# Enable Change Data Feed
spark.sql(f"""
    ALTER TABLE {TARGET_TABLE}
    SET TBLPROPERTIES (
        delta.enableChangeDataFeed = true
    )
""")

In [0]:
# ============================================================
# 8. INCREMENTAL SYNCHRONIZATION
# ============================================================

if not spark.catalog.tableExists(TARGET_TABLE):

    raise RuntimeError("Target table creation failed.")

# Use the latest complete snapshot of the eight documents.
# Delete missing chunks, update changed chunks, insert new chunks.

delta_table = DeltaTable.forName(spark, TARGET_TABLE)

(
    delta_table.alias("target")
    .merge(
        chunks_df.alias("source"),
        "target.chunk_id = source.chunk_id"
    )
    .whenMatchedUpdate(
        condition="""
            NOT (
                target.doc_name <=> source.doc_name
                AND target.heading <=> source.heading
                AND target.chunk_index <=> source.chunk_index
                AND target.text <=> source.text
            )
        """,
        set={
            "doc_name": "source.doc_name",
            "heading": "source.heading",
            "chunk_index": "source.chunk_index",
            "text": "source.text"
        }
    )
    .whenNotMatchedInsert(
        values={
            "chunk_id": "source.chunk_id",
            "doc_name": "source.doc_name",
            "heading": "source.heading",
            "chunk_index": "source.chunk_index",
            "text": "source.text"
        }
    )
    .whenNotMatchedBySourceDelete()
    .execute()
)


print("Policy chunk table synchronized successfully.")

In [0]:
%sql
SELECT
    COUNT(*) AS total_chunks,
    COUNT(DISTINCT doc_name) AS total_documents,
    COUNT(DISTINCT chunk_id) AS unique_chunk_ids
FROM claims.genai.policy_chunks;

In [0]:
%sql
SELECT
    version,
    timestamp,
    operation,
    operationMetrics
FROM (
    DESCRIBE HISTORY claims.genai.policy_chunks
)
WHERE operation = 'MERGE'
ORDER BY version DESC;

In [0]:
%sql
SELECT
    COUNT(*) AS total_chunks,
    COUNT(DISTINCT chunk_id) AS unique_chunks,
    COUNT_IF(text IS NULL OR TRIM(text) = '') AS empty_chunks
FROM claims.genai.policy_chunks;

### R2 - Create the Delta Sync index

In [0]:
%pip install databricks-vectorsearch databricks-ai-search databricks-langchain langchain-openai langchain-core pydantic

In [0]:
dbutils.library.restartPython()

In [0]:
# Create the endpoint
from databricks.ai_search.client import AISearchClient

client = AISearchClient()

client.create_endpoint(
    name="claims_vs",
    endpoint_type="STANDARD"
)

print("AI Search endpoint creation requested.")

In [0]:
# Create the index
index = client.create_delta_sync_index(
    endpoint_name="claims_vs",
    index_name="claims.genai.policy_chunks_idx",
    source_table_name="claims.genai.policy_chunks",
    pipeline_type="TRIGGERED",
    primary_key="chunk_id",
    embedding_source_column="text",
    embedding_model_endpoint_name="databricks-gte-large-en"
)

print("Delta Sync index creation requested.")

In [0]:
index = client.get_index(
    endpoint_name="claims_vs",
    index_name="claims.genai.policy_chunks_idx"
)

index.sync()

In [0]:
from databricks_langchain import ChatDatabricks
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

In [0]:
# Credentials come from the notebook session automatically — never hardcode tokens
from databricks.sdk import WorkspaceClient

_w = WorkspaceClient()
_auth_headers = _w.config.authenticate()

workspace_url = _w.config.host.rstrip("/")
workspace_token = _auth_headers["Authorization"].removeprefix("Bearer ")

LLM_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"

In [0]:
chat_databricks = ChatDatabricks(
    endpoint=LLM_ENDPOINT,
    temperature=0
)

response_db = chat_databricks.invoke([
    HumanMessage(
        content="Explain what a health insurance waiting period means in two sentences."
    )
])

chat_databricks_reply = response_db.content

print(chat_databricks_reply)

In [0]:
chat_openai = ChatOpenAI(
    model=LLM_ENDPOINT,
    base_url=f"{workspace_url}/serving-endpoints",
    api_key=workspace_token,
    temperature=0
)

response_openai = chat_openai.invoke([
    HumanMessage(
        content="Explain what a health insurance waiting period means in two sentences."
    )
])

chat_openai_reply = response_openai.content

print(chat_openai_reply)

In [0]:
# Define the typed Pydantic output schema
from pydantic import BaseModel, Field

class WaitingPeriodFacts(BaseModel):

    document_name: str = Field(
        description="Exact policy document filename"
    )

    plan_family: str = Field(
        description="Insurance plan family described in the document"
    )

    initial_waiting_days: int = Field(
        ge=0,
        description="Initial waiting period in days"
    )

    pre_existing_disease_waiting_months: int = Field(
        ge=0,
        description="Pre-existing disease waiting period in months"
    )

In [0]:
# Create the extraction chain
from langchain_core.prompts import ChatPromptTemplate

structured_llm = chat_databricks.with_structured_output(
    WaitingPeriodFacts,
    method="function_calling"
)

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        You are an insurance policy document extraction assistant.

        Extract waiting-period information only from the supplied policy document.

        Return these fields:
        1. document_name
        2. plan_family
        3. initial_waiting_days
        4. pre_existing_disease_waiting_months

        Rules:
        - Use only information explicitly supported by the document.
        - Do not guess or use general insurance knowledge.
        - Extract the initial waiting period in days.
        - Extract the pre-existing disease waiting period in months.
        - If the document expresses a duration in years, convert it to months.
        - Do not confuse specific disease waiting periods with the
          pre-existing disease waiting period.
        - Preserve the exact document filename.
        """
    ),
    (
        "human",
        """
        Document filename:
        {document_name}

        Policy document content:
        {document_text}

        Extract the waiting-period facts.
        """
    )
])

extraction_chain = prompt | structured_llm

In [0]:
# finding documents begin with 02 and 03
import os
from pathlib import Path

POLICY_PATH = "/Volumes/claims/raw/landing/genai/policy_docs/"

files = os.listdir(POLICY_PATH)

doc_02 = next(
    (f for f in files if f.startswith("02") and f.endswith(".md")),
    None
)

doc_03 = next(
    (f for f in files if f.startswith("03") and f.endswith(".md")),
    None
)

if not doc_02 or not doc_03:
    raise FileNotFoundError(
        f"Could not find both waiting-period documents. "
        f"Found: {doc_02}, {doc_03}"
    )

print("Document 02:", doc_02)
print("Document 03:", doc_03)

In [0]:
# Read the document contents:
def read_policy_document(filename):

    file_path = os.path.join(POLICY_PATH, filename)

    with open(file_path, "r", encoding="utf-8") as file:
        return file.read()


document_02_text = read_policy_document(doc_02)
document_03_text = read_policy_document(doc_03)

print("Document 02 characters:", len(document_02_text))
print("Document 03 characters:", len(document_03_text))

In [0]:
# Extract structured facts from both documents
waiting_facts = []

for filename, document_text in [
    (doc_02, document_02_text),
    (doc_03, document_03_text)
]:

    extracted = extraction_chain.invoke({
        "document_name": filename,
        "document_text": document_text
    })

    # Ensure the exact source filename is retained.
    extracted = extracted.model_copy(
        update={"document_name": filename}
    )

    waiting_facts.append(extracted.model_dump())

    print(f"\nExtracted from: {filename}")
    print(extracted.model_dump())

In [0]:
assert len(waiting_facts) == 2

assert waiting_facts[0]["document_name"] == doc_02
assert waiting_facts[1]["document_name"] == doc_03

for fact in waiting_facts:
    assert isinstance(fact["initial_waiting_days"], int)
    assert isinstance(
        fact["pre_existing_disease_waiting_months"], int
    )

In [0]:
import json
import os
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.workspace import ImportFormat

# Final evidence object
evidence = {
    "chat_databricks_reply": chat_databricks_reply,
    "chat_openai_reply": chat_openai_reply,
    "waiting_facts": waiting_facts
}

# Validate non-empty chat replies
assert isinstance(chat_databricks_reply, str)
assert chat_databricks_reply.strip() != ""

assert isinstance(chat_openai_reply, str)
assert chat_openai_reply.strip() != ""

w = WorkspaceClient()

output_path = (
    "/Workspace/Users/surajitm0nd0l12@gmail.com/"
    "Saffron_Shield/evidence/r3_langchain.json"
)

json_content = json.dumps(
    evidence,
    indent=4,
    ensure_ascii=False
)

# Ensure the parent folder exists (upload does not create folders)
w.workspace.mkdirs(os.path.dirname(output_path))

w.workspace.upload(
    output_path,
    json_content.encode("utf-8"),
    format=ImportFormat.AUTO,
    overwrite=True
)

print("Evidence uploaded:", output_path)

### R4 - Hybrid retriever & rerank

In [0]:
from databricks.ai_search.client import AISearchClient
from databricks.ai_search.reranker import DatabricksReranker

client = AISearchClient()

index = client.get_index(
    endpoint_name="claims_vs",
    index_name="claims.genai.policy_chunks_idx"
)

In [0]:
def retrieve_policy_chunks(question, top_k=5):

    results = index.similarity_search(
        query_text=question,
        columns=[
            "chunk_id",
            "doc_name",
            "heading",
            "chunk_index",
            "text"
        ],
        num_results=top_k,
        query_type="hybrid",
        reranker=DatabricksReranker(
            columns_to_rerank=["heading", "text"]
        ),
        debug_level=1
    )

    return results


# Test with an individual-plan question
test_question = (
    "What are the initial and pre-existing disease waiting periods "
    "for individual health insurance plans?"
)

test_results = retrieve_policy_chunks(test_question)

print(test_results)

In [0]:
%pip show databricks-ai-search